<a href="https://colab.research.google.com/github/ElofssonLab/kb8029-book/blob/main/notebooks/day11-discussion-2.ipynb" style="display:inline-block;padding:10px 18px;background-color:#F9AB00;color:#000000;font-weight:bold;text-decoration:none;border-radius:6px;font-family:sans-serif;font-size:14px;">&#9654;&nbsp; Open in Google Colab</a>

# Day 11 — Discussion 2: do bacterial signal peptides need bidirectional context as much as the main dataset? {.unnumbered}

The main notebook's case study pools reviewed UniProt entries across
all organisms. Bacterial (Sec/SPI) signal peptides follow a
well-known, quite rigid grammar (positively charged n-region,
hydrophobic h-region, small residues right before the cleavage site)
— this notebook asks whether that more rigid grammar makes BiLSTM's
advantage over a forward-only LSTM smaller on a bacteria-only subset
than it is on the general dataset, or whether the much smaller,
more class-imbalanced bacterial sample makes the gap even bigger
instead.

In [1]:
import re
import urllib.parse
import urllib.request
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import f1_score, matthews_corrcoef, roc_auc_score

torch.manual_seed(0)
np.random.seed(0)

AA = "ACDEFGHIKLMNPQRSTVWY"
aa_to_idx = {a: i for i, a in enumerate(AA)}
UNK_IDX = len(AA)
VOCAB_SIZE = len(AA) + 1

def aa_index(ch):
    return aa_to_idx.get(ch, UNK_IDX)

def fetch_uniprot_tsv(query, fields, max_records=1200):
    base = "https://rest.uniprot.org/uniprotkb/search"
    params = {"query": query, "fields": fields, "format": "tsv", "size": 500}
    next_url = base + "?" + urllib.parse.urlencode(params)
    chunks, n_rows = [], 0
    while next_url and n_rows < max_records:
        req = urllib.request.Request(next_url, headers={"User-Agent": "EkmanTeaching/1.0"})
        with urllib.request.urlopen(req, timeout=60) as resp:
            text = resp.read().decode("utf-8", errors="ignore")
            lines = text.strip().splitlines()
            if not lines:
                break
            if not chunks:
                chunks.extend(lines); n_rows += max(0, len(lines) - 1)
            else:
                chunks.extend(lines[1:]); n_rows += len(lines) - 1
            link = resp.headers.get("Link", "")
            m = re.search(r"<([^>]+)>;\s*rel=\"next\"", link)
            next_url = m.group(1) if m else None
    return "\n".join(chunks) + "\n"

def parse_signal_ranges(signal_field):
    if signal_field is None:
        return []
    txt = str(signal_field).strip()
    if txt == "" or txt.lower() == "nan":
        return []
    ranges = []
    for m in re.finditer(r"SIGNAL\s+(\d+)\.\.(\d+)", txt):
        ranges.append((int(m.group(1)), int(m.group(2))))
    return ranges

def load_dataset(query, fields="accession,sequence,ft_signal,organism_name", max_records=1200):
    tsv_text = fetch_uniprot_tsv(query, fields, max_records)
    rows = [r.split("\t") for r in tsv_text.strip().splitlines()]
    header, data = rows[0], rows[1:]
    sequences, labels = [], []
    for row in data:
        rec = dict(zip(header, row))
        seq = rec.get("Sequence", "")
        if not seq:
            continue
        ranges = parse_signal_ranges(rec.get("Signal peptide"))
        yseq = [0] * len(seq)
        for (a, b) in ranges:
            for p in range(a - 1, min(b, len(seq))):
                yseq[p] = 1
        sequences.append(seq)
        labels.append(yseq)
    return sequences, labels

def make_seq_tensors(seq_list, lab_list, nterm_len=70):
    X = np.full((len(seq_list), nterm_len), UNK_IDX, dtype=np.int64)
    Y = np.zeros((len(seq_list), nterm_len), dtype=np.float32)
    M = np.zeros((len(seq_list), nterm_len), dtype=np.float32)
    for i, (seq, yseq) in enumerate(zip(seq_list, lab_list)):
        L = min(len(seq), nterm_len)
        for j in range(L):
            X[i, j] = aa_index(seq[j])
            Y[i, j] = float(yseq[j])
            M[i, j] = 1.0
    return X, Y, M

class SeqTagger(nn.Module):
    def __init__(self, vocab_size, emb_dim=24, hid_dim=48, kind="lstm", num_layers=1):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, emb_dim)
        bidirectional = (kind == "bilstm")
        if kind in ("lstm", "bilstm"):
            self.rnn = nn.LSTM(emb_dim, hid_dim, num_layers=num_layers, batch_first=True, bidirectional=bidirectional)
        elif kind == "gru":
            self.rnn = nn.GRU(emb_dim, hid_dim, num_layers=num_layers, batch_first=True)
        else:
            self.rnn = nn.RNN(emb_dim, hid_dim, num_layers=num_layers, batch_first=True, nonlinearity="tanh")
        out_dim = hid_dim * (2 if bidirectional else 1)
        self.out = nn.Linear(out_dim, 1)

    def forward(self, x):
        z = self.emb(x)
        h, _ = self.rnn(z)
        return self.out(h).squeeze(-1)

def n_params(model):
    return sum(p.numel() for p in model.parameters())

def train_model(model, train_loader, epochs=12, lr=1e-2):
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    lossf = nn.BCEWithLogitsLoss(reduction="none")
    for ep in range(epochs):
        model.train()
        for Xb, Yb, Mb in train_loader:
            opt.zero_grad()
            logits = model(Xb)
            loss = (lossf(logits, Yb) * Mb).sum() / Mb.sum()
            loss.backward()
            opt.step()
    return model

@torch.no_grad()
def evaluate(model, X_test, Y_test, M_test):
    model.eval()
    logits = model(torch.from_numpy(X_test))
    proba = torch.sigmoid(logits).numpy()
    mask = M_test.astype(bool)
    y_true = Y_test[mask]
    p = proba[mask]
    pred = (p >= 0.5).astype(np.int64)
    return {
        "f1": f1_score(y_true, pred, zero_division=0),
        "mcc": matthews_corrcoef(y_true, pred),
        "auroc": roc_auc_score(y_true, p),
    }


## 1) A real bacteria-only UniProt fetch

In [2]:

seq_bact, lab_bact = load_dataset(
    "reviewed:true AND length:[40 TO 800] AND taxonomy_id:2",  # 2 = Bacteria
    max_records=900,
)
print(f"Bacterial entries fetched: {len(seq_bact)}")
pos_rate = np.mean([sum(y) > 0 for y in lab_bact])
print(f"Fraction with an annotated signal peptide: {pos_rate:.3f}")


Bacterial entries fetched: 1000
Fraction with an annotated signal peptide: 0.083


## 2) LSTM vs. BiLSTM on this bacteria-only subset

In [3]:

idx = np.arange(len(seq_bact))
rng = np.random.default_rng(0)
rng.shuffle(idx)
split = int(0.8 * len(idx))
seq_train = [seq_bact[i] for i in idx[:split]]; lab_train = [lab_bact[i] for i in idx[:split]]
seq_test = [seq_bact[i] for i in idx[split:]]; lab_test = [lab_bact[i] for i in idx[split:]]

X_train, Y_train, M_train = make_seq_tensors(seq_train, lab_train)
X_test, Y_test, M_test = make_seq_tensors(seq_test, lab_test)
train_loader = DataLoader(TensorDataset(torch.from_numpy(X_train), torch.from_numpy(Y_train), torch.from_numpy(M_train)), batch_size=64, shuffle=True)

for kind in ["lstm", "bilstm"]:
    torch.manual_seed(0)
    model = SeqTagger(VOCAB_SIZE, kind=kind)
    train_model(model, train_loader, epochs=12)
    metrics = evaluate(model, X_test, Y_test, M_test)
    print(f"{kind:8s} ({n_params(model):,} params) -> F1={metrics['f1']:.3f} "
          f"MCC={metrics['mcc']:.3f} AUROC={metrics['auroc']:.3f}")


lstm     (14,761 params) -> F1=0.000 MCC=0.000 AUROC=0.946


bilstm   (29,017 params) -> F1=0.508 MCC=0.559 AUROC=0.983


## Try it yourself

The real measured gap here (BiLSTM F1=0.508 vs. LSTM F1=0.000 — a
forward-only LSTM that collapsed to predicting no positives at all)
is far more dramatic than the main notebook's general-dataset gap
(F1 0.590 vs. 0.361). That is the opposite of this notebook's opening
guess: a more rigid grammar did not make bidirectional context less
necessary, it made this small, imbalanced sample harder for a
forward-only model to learn from at all. Try taxonomy_id:2759
(Eukaryota) instead of taxonomy_id:2 (Bacteria) and compare a third
time -- does a larger, better-balanced eukaryotic sample let plain
LSTM recover, or does it collapse the same way?